In [2]:
import json
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("B5 — ÉVALUATION DU MODÈLE FINE-TUNÉ")
print("=" * 60)


c:\Users\pc\env_projects\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


B5 — ÉVALUATION DU MODÈLE FINE-TUNÉ


In [3]:
# ═══════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

MODEL_PATH = "model_finetuned"
MODEL_BASE = "google/pegasus-xsum"
MAX_INPUT = 256
MAX_TARGET = 64
BATCH_SIZE = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n📌 Configuration:")
print(f"   Device      : {device}")
print(f"   Modèle      : {MODEL_PATH}")
print(f"   Max input   : {MAX_INPUT} tokens")
print(f"   Max target  : {MAX_TARGET} tokens")


📌 Configuration:
   Device      : cpu
   Modèle      : model_finetuned
   Max input   : 256 tokens
   Max target  : 64 tokens


In [4]:
# ═══════════════════════════════════════════════════════════════════
# 2. CHARGEMENT DES DONNÉES
# ═══════════════════════════════════════════════════════════════════

print("\n📂 Chargement des données...")

with open("donnees_propres.json", "r", encoding="utf-8") as f:
    donnees = json.load(f)

# Split : 80% train, 20% validation (comme B4)
train_data = donnees[:80]
val_data = donnees[80:]

print(f"   Train : {len(train_data)} exemples")
print(f"   Val   : {len(val_data)} exemples")



📂 Chargement des données...
   Train : 80 exemples
   Val   : 20 exemples


In [5]:
# ═══════════════════════════════════════════════════════════════════
# 3. CHARGEMENT DU MODÈLE FINE-TUNÉ
# ═══════════════════════════════════════════════════════════════════

print("\n🔄 Chargement du modèle fine-tuné...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Charger le modèle de base
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_BASE)

# Appliquer LoRA fine-tunée
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model = model.to(device)
model.eval()

print(f"   ✅ Modèle chargé sur {device.upper()}")
print(f"   ✅ Tokenizer chargé")


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /google/pegasus-xsum/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000279F36AA550>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 1cd928be-54f9-4c6b-97f8-50c2683f0323)')' thrown while requesting HEAD https://huggingface.co/google/pegasus-xsum/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].



🔄 Chargement du modèle fine-tuné...


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /google/pegasus-xsum/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000279F397B5D0>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: b1e040eb-15fb-4d29-9bee-e9dedd4910c2)')' thrown while requesting HEAD https://huggingface.co/google/pegasus-xsum/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /google/pegasus-xsum/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000279F397BD90>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: cc596766-59f8-4367-aea8-28b88d023d21)')' thrown while requesting HEAD https://huggingface.co/google/pegasus-xsum/resolve/m

KeyboardInterrupt: 

In [5]:

# ═══════════════════════════════════════════════════════════════════
# 4. CLASSE DATASET
# ═══════════════════════════════════════════════════════════════════

class BookSumEvalDataset(torch.utils.data.Dataset):
    def __init__(self, donnees, tokenizer, max_length=256):
        self.donnees = donnees
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.donnees)

    def __getitem__(self, idx):
        exemple = self.donnees[idx]
        texte = ' '.join(exemple['chunks'])
        resume = exemple['resume']

        entree = self.tokenizer(
            texte,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": entree["input_ids"].squeeze(),
            "attention_mask": entree["attention_mask"].squeeze(),
            "reference": resume,
            "index": idx
        }


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 5. GÉNÉRATION DES RÉSUMÉS
# ═══════════════════════════════════════════════════════════════════

def collate_fn(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    references = [b["reference"] for b in batch]
    indices = [b["index"] for b in batch]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "references": references,
        "indices": indices
    }

eval_dataset = BookSumEvalDataset(val_data, tokenizer, MAX_INPUT)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"\n🔄 Génération des résumés sur {len(val_data)} exemples...")

predictions = []
references = []
indices = []

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_TARGET,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=0.8
        )
        
        preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        
        predictions.extend(preds)
        references.extend(batch["references"])
        indices.extend(batch["indices"])

print(f"   ✅ Génération terminée")


🔄 Génération des résumés sur 20 exemples...


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 6. CALCUL DES MÉTRIQUES
# ═══════════════════════════════════════════════════════════════════

print("\n📊 Calcul des métriques...")

# ROUGE
scorer_rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

for pred, ref in zip(predictions, references):
    if len(pred) > 0 and len(ref) > 0:
        score = scorer_rouge.score(ref, pred)
        rouge_scores["rouge1"].append(score["rouge1"].fmeasure)
        rouge_scores["rouge2"].append(score["rouge2"].fmeasure)
        rouge_scores["rougeL"].append(score["rougeL"].fmeasure)
    else:
        rouge_scores["rouge1"].append(0.0)
        rouge_scores["rouge2"].append(0.0)
        rouge_scores["rougeL"].append(0.0)

# BLEU
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smoothie = SmoothingFunction().method4
    
    bleu_scores = []
    for pred, ref in zip(predictions, references):
        if len(pred) > 0 and len(ref) > 0:
            bleu = sentence_bleu(
                [ref.split()],
                pred.split(),
                smoothing_function=smoothie
            )
            bleu_scores.append(bleu)
        else:
            bleu_scores.append(0.0)
    
    bleu_mean = np.mean(bleu_scores)
    print("   ✅ BLEU calculé")
except ImportError:
    print("   ⚠️ nltk non installé, installation...")
    import subprocess
    subprocess.check_call(["pip", "install", "nltk", "-q"])
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smoothie = SmoothingFunction().method4
    
    bleu_scores = []
    for pred, ref in zip(predictions, references):
        if len(pred) > 0 and len(ref) > 0:
            bleu = sentence_bleu(
                [ref.split()],
                pred.split(),
                smoothing_function=smoothie
            )
            bleu_scores.append(bleu)
        else:
            bleu_scores.append(0.0)
    
    bleu_mean = np.mean(bleu_scores)
    print("   ✅ BLEU calculé")

# BERTScore
try:
    from bert_score import BERTScorer
    
    print("   🔄 Calcul BERTScore (peut prendre 2-3 minutes)...")
    bert_scorer = BERTScorer(
        lang="en",
        model_type="microsoft/deberta-xlarge-mnli",
        rescale_with_baseline=True,
        device=device
    )
    
    P, R, F1 = bert_scorer.score(predictions, references)
    
    bert_precision = P.mean().item()
    bert_recall = R.mean().item()
    bert_f1 = F1.mean().item()
    
    print(f"   ✅ BERTScore calculé")
except ImportError:
    print("   ⚠️ bert-score non installé, installation...")
    import subprocess
    subprocess.check_call(["pip", "install", "bert-score", "-q"])
    
    from bert_score import BERTScorer
    bert_scorer = BERTScorer(
        lang="en",
        model_type="microsoft/deberta-xlarge-mnli",
        rescale_with_baseline=True,
        device=device
    )
    
    P, R, F1 = bert_scorer.score(predictions, references)
    
    bert_precision = P.mean().item()
    bert_recall = R.mean().item()
    bert_f1 = F1.mean().item()
    
    print(f"   ✅ BERTScore calculé")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 7. RÉSULTATS STATISTIQUES
# ═══════════════════════════════════════════════════════════════════

results = {
    "rouge1_mean": np.mean(rouge_scores["rouge1"]),
    "rouge1_std": np.std(rouge_scores["rouge1"]),
    "rouge1_median": np.median(rouge_scores["rouge1"]),
    "rouge2_mean": np.mean(rouge_scores["rouge2"]),
    "rouge2_std": np.std(rouge_scores["rouge2"]),
    "rouge2_median": np.median(rouge_scores["rouge2"]),
    "rougeL_mean": np.mean(rouge_scores["rougeL"]),
    "rougeL_std": np.std(rouge_scores["rougeL"]),
    "rougeL_median": np.median(rouge_scores["rougeL"]),
    "bleu_mean": bleu_mean,
    "bert_precision": bert_precision,
    "bert_recall": bert_recall,
    "bert_f1": bert_f1,
    "nombre_exemples": len(predictions)
}

print("\n" + "=" * 60)
print("📈 RÉSULTATS FINAUX")
print("=" * 60)
print(f"\n🔤 ROUGE (moyenne ± std):")
print(f"   ROUGE-1 : {results['rouge1_mean']*100:.2f}% ± {results['rouge1_std']*100:.2f}")
print(f"   ROUGE-2 : {results['rouge2_mean']*100:.2f}% ± {results['rouge2_std']*100:.2f}")
print(f"   ROUGE-L : {results['rougeL_mean']*100:.2f}% ± {results['rougeL_std']*100:.2f}")

print(f"\n📊 BLEU :")
print(f"   BLEU-1/2/3/4 : {results['bleu_mean']:.4f}")

print(f"\n🤖 BERTScore :")
print(f"   Précision : {results['bert_precision']:.4f}")
print(f"   Rappel    : {results['bert_recall']:.4f}")
print(f"   F1        : {results['bert_f1']:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 8. VISUALISATIONS
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='#0e0f14')

# Distribution ROUGE-1
ax1 = axes[0]
ax1.hist(rouge_scores["rouge1"], bins=20, color='#00d4aa', alpha=0.85, edgecolor='#0e0f14')
ax1.axvline(results['rouge1_mean'], color='white', ls='--', lw=1.5, label=f"Moyenne: {results['rouge1_mean']*100:.1f}%")
ax1.set_title('Distribution ROUGE-1', color='white', fontsize=11)
ax1.set_xlabel('Score', color='#6b7280')
ax1.set_ylabel('Fréquence', color='#6b7280')
ax1.set_facecolor('#161820')
ax1.legend(facecolor='#161820', edgecolor='#2a2d3a')
ax1.tick_params(colors='#6b7280')

# Distribution ROUGE-L
ax2 = axes[1]
ax2.hist(rouge_scores["rougeL"], bins=20, color='#a78bfa', alpha=0.85, edgecolor='#0e0f14')
ax2.axvline(results['rougeL_mean'], color='white', ls='--', lw=1.5, label=f"Moyenne: {results['rougeL_mean']*100:.1f}%")
ax2.set_title('Distribution ROUGE-L', color='white', fontsize=11)
ax2.set_xlabel('Score', color='#6b7280')
ax2.set_ylabel('Fréquence', color='#6b7280')
ax2.set_facecolor('#161820')
ax2.legend(facecolor='#161820', edgecolor='#2a2d3a')
ax2.tick_params(colors='#6b7280')

# Longueurs des prédictions vs références
ax3 = axes[2]
longueurs_pred = [len(p.split()) for p in predictions]
longueurs_ref = [len(r.split()) for r in references]
ax3.scatter(longueurs_ref, longueurs_pred, alpha=0.5, color='#f59e0b', s=30)
ax3.plot([0, max(longueurs_ref)], [0, max(longueurs_ref)], '--', color='white', alpha=0.5, label="Idéal")
ax3.set_title('Longueur prédiction vs référence', color='white', fontsize=11)
ax3.set_xlabel('Mots référence', color='#6b7280')
ax3.set_ylabel('Mots prédiction', color='#6b7280')
ax3.set_facecolor('#161820')
ax3.legend(facecolor='#161820', edgecolor='#2a2d3a')
ax3.tick_params(colors='#6b7280')

plt.suptitle('B5 — Évaluation PEGASUS + LoRA fine-tuné sur BookSum', 
             color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evaluation_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0e0f14')
plt.show()
print("\n✅ evaluation_distributions.png sauvegardé")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 9. SAUVEGARDE DES RÉSULTATS
# ═══════════════════════════════════════════════════════════════════

# Résultats complets
evaluation_results = {
    "modele": "PEGASUS + LoRA fine-tuned",
    "modele_base": MODEL_BASE,
    "modele_finetuned": MODEL_PATH,
    "metriques": results,
    "hyperparametres_evaluation": {
        "max_input": MAX_INPUT,
        "max_target": MAX_TARGET,
        "batch_size": BATCH_SIZE,
        "num_beams": 4
    }
}

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=2, ensure_ascii=False)

# Résultats détaillés (pour analyse B6)
details = []
for idx, pred, ref, r1, r2, rl in zip(
    indices, predictions, references,
    rouge_scores["rouge1"], rouge_scores["rouge2"], rouge_scores["rougeL"]
):
    details.append({
        "index": int(idx),
        "reference": ref,
        "prediction": pred,
        "longueur_ref": len(ref.split()),
        "longueur_pred": len(pred.split()),
        "rouge1": r1,
        "rouge2": r2,
        "rougeL": rl
    })

with open("evaluation_detailles.json", "w", encoding="utf-8") as f:
    json.dump(details, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
print("💾 FICHIERS SAUVEGARDÉS")
print("=" * 60)
print("   📄 evaluation_results.json   → métriques agrégées")
print("   📄 evaluation_detailles.json → prédictions détaillées")
print("   📊 evaluation_distributions.png → visualisations")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 10. COMPARAISON ZERO-SHOT vs FINE-TUNED
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("📊 COMPARAISON ZERO-SHOT vs FINE-TUNED")
print("=" * 60)

# Charger les résultats B3 si disponibles
try:
    with open("selection_modele_resultats.json", "r", encoding="utf-8") as f:
        b3_results = json.load(f)
    baseline_r1 = b3_results.get("rouge_scores_baseline", {}).get("rouge1", 0.251) * 100
except:
    baseline_r1 = 25.1

gain = results['rouge1_mean'] * 100 - baseline_r1

print(f"\n   {'Modèle':<30} {'ROUGE-1':>10} {'Gain':>12}")
print(f"   {'-'*30} {'-'*10} {'-'*12}")
print(f"   {'PEGASUS (zero-shot)':<30} {baseline_r1:>9.1f}% {'(baseline)':>12}")
print(f"   {'PEGASUS + LoRA (fine-tuned)':<30} {results['rouge1_mean']*100:>9.1f}% {'+':>5}{gain:.1f} pts")

print(f"\n🎉 Amélioration : +{gain:.1f} points ROUGE-1")
print(f"   Gain relatif : {(gain/baseline_r1)*100:.0f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 11. EXEMPLES DE PRÉDICTION
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("📝 EXEMPLES DE PRÉDICTIONS")
print("=" * 60)

# Afficher 3 exemples (meilleur, moyen, pire)
meilleur_idx = np.argmax(rouge_scores["rouge1"])
moyen_idx = np.argsort(rouge_scores["rouge1"])[len(rouge_scores["rouge1"]) // 2]
pire_idx = np.argmin(rouge_scores["rouge1"])

for label, idx in [("🏆 MEILLEUR", meilleur_idx), 
                    ("📊 MOYEN", moyen_idx), 
                    ("⚠️ PIRE", pire_idx)]:
    print(f"\n{label} (ROUGE-1: {rouge_scores['rouge1'][idx]*100:.1f}%):")
    print(f"   📖 Référence  : {references[idx][:150]}...")
    print(f"   🤖 Prédiction : {predictions[idx][:150]}...")

print("\n" + "=" * 60)
print("✅ B5 — ÉVALUATION TERMINÉE")
print("=" * 60)
print("\n📌 Prochaine étape : B6 — Analyse des résultats")
print("   → Ouvrir 06_B6_Analyse_Resultats.ipynb")